In [17]:
import os
# import torch
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import pandas as pd

def extract_loss_functions_from_pytorch_experiments(runs_dir):
    """
    Извлекает информацию о функциях потерь из экспериментов PyTorch TensorBoard
    
    Args:
        runs_dir (str): Путь к папке с экспериментами
    
    Returns:
        dict: Словарь с информацией о функциях потерь для каждого эксперимента
    """
    experiments_data = {}
    
    # Проверяем существование папки
    if not os.path.exists(runs_dir):
        print(f"❌ Папка {runs_dir} не существует")
        return experiments_data
    
    # Проходим по всем подпапкам в runs
    for experiment_name in os.listdir(runs_dir):
        experiment_path = os.path.join(runs_dir, experiment_name)
        
        # Пропускаем файлы, только папки
        if not os.path.isdir(experiment_path):
            continue
            
        print(f"🔍 Анализируем эксперимент: {experiment_name}")
        
        # Ищем event файлы в папке эксперимента
        event_files = [f for f in os.listdir(experiment_path) if f.startswith("events.out.tfevents")]
        
        if not event_files:
            print(f"  ⚠️ В эксперименте {experiment_name} не найдены event файлы")
            continue
        
        # Берем первый event файл (обычно он один)
        event_file_path = os.path.join(experiment_path, event_files[0])
        
        try:
            # Создаем EventAccumulator с правильными настройками для PyTorch
            event_acc = EventAccumulator(
                event_file_path,
                size_guidance={
                    'tensors': 0,  # не загружаем тензоры
                    'text': 10,    # загружаем до 10 текстовых элементов
                }
            )
            event_acc.Reload()
            
            # Получаем список всех тегов
            tags = event_acc.Tags()
            print(f"  📋 Найденные теги: {list(tags.keys())}")
            
            # Ищем текстовые теги
            if 'text' in tags:
                print(f"  📝 Текстовые теги: {tags['text']}")
                
                # Пробуем разные варианты названия тега
                possible_tags = [
                    'Training/Loss_Function',
                    'loss_function',
                    'Loss_Function',
                    'loss_func',
                    'model/loss_function'
                ]
                
                found_tag = None
                for tag in possible_tags:
                    if tag in tags['text']:
                        found_tag = tag
                        break
                
                if found_tag:
                    # Извлекаем текстовые данные
                    text_events = event_acc.Text(found_tag)
                    
                    if text_events:
                        # Берем последнее значение
                        loss_function_info = text_events[-1].text.decode('utf-8')
                        
                        experiments_data[experiment_name] = {
                            'experiment_path': experiment_path,
                            'loss_function_info': loss_function_info,
                            'timestamp': text_events[-1].wall_time,
                            'tag_used': found_tag
                        }
                        print(f"  ✅ Найдена информация о функции потерь (тег: {found_tag})")
                    else:
                        print(f"  ⚠️ Тег '{found_tag}' найден, но данных нет")
                else:
                    print(f"  🔍 Поиск по альтернативным тегам: {possible_tags}")
                    # Выводим все доступные текстовые теги
                    print(f"  📋 Все текстовые теги в эксперименте: {tags['text']}")
            else:
                print(f"  ❌ Текстовые теги не найдены в эксперименте")
                
        except Exception as e:
            print(f"  ❌ Ошибка при чтении {experiment_name}: {e}")
    
    return experiments_data

def extract_all_text_data_from_experiment(experiment_path):
    """
    Извлекает все текстовые данные из эксперимента для отладки
    """
    event_files = [f for f in os.listdir(experiment_path) if f.startswith("events.out.tfevents")]
    
    if not event_files:
        return {}
    
    event_file_path = os.path.join(experiment_path, event_files[0])
    
    try:
        event_acc = EventAccumulator(event_file_path)
        event_acc.Reload()
        
        tags = event_acc.Tags()
        all_text_data = {}
        
        if 'text' in tags:
            for tag in tags['text']:
                text_events = event_acc.Text(tag)
                if text_events:
                    all_text_data[tag] = text_events[-1].text.decode('utf-8')
        
        return all_text_data
    except Exception as e:
        print(f"Ошибка при чтении {experiment_path}: {e}")
        return {}

# Альтернативный способ через прямой парсинг event файлов
def find_all_text_tags_in_runs(runs_dir):
    """
    Находит все текстовые теги во всех экспериментах
    """
    all_tags = set()
    
    for experiment_name in os.listdir(runs_dir):
        experiment_path = os.path.join(runs_dir, experiment_name)
        
        if not os.path.isdir(experiment_path):
            continue
            
        event_files = [f for f in os.listdir(experiment_path) if f.startswith("events.out.tfevents")]
        
        if event_files:
            event_file_path = os.path.join(experiment_path, event_files[0])
            
            try:
                event_acc = EventAccumulator(event_file_path)
                event_acc.Reload()
                
                tags = event_acc.Tags()
                if 'tensors' in tags:
                    loss_tags = [tag for tag in tags['tensors'] if 'Loss_Function' in tag]
                    for tag in loss_tags:
                        tensor_events = event_acc.Tensors(tag)
                        if tensor_events:
                            # Декодируем tensor данные как текст
                            tensor_data = tensor_events[-1].tensor_proto.string_val[0]
                            loss_function_info = tensor_data.decode('utf-8')
                            # print(f"✅ Найдена функция потерь: {loss_function_info}")
                            loss_func_before, loss_func_after = extract_loss_functions_from_tensor(loss_function_info)
                            print(loss_func_before)
                            print("="*100)
                            print(loss_func_after)
                            print("\n\n")
        
                    
            except Exception as e:
                print(f"Ошибка в {experiment_name}: {e}")
    
    print("🎯 Все найденные текстовые теги:")
    for tag in sorted(all_tags):
        print(f"  - {tag}")
    
    return all_tags


# Основная функция для использования
def main():
    print("🔍 Поиск функций потерь в экспериментах PyTorch...")
    
    # Сначала найдем все теги для отладки
    print("\n📋 Поиск всех текстовых тегов...")
    all_tags = find_all_text_tags_in_runs(runs_dir="../runs/test/1/DeepSeek-V3.1")
    
    # Затем извлечем данные
    print("\n📥 Извлечение данных...")
    experiments_data = extract_loss_functions_from_pytorch_experiments(runs_dir="../runs/test/1/DeepSeek-V3.1")
    
    if experiments_data:
        print(f"\n✅ Найдено {len(experiments_data)} экспериментов с функциями потерь")
        
        
        # Выводим краткую информацию
        print("\n📊 Краткий обзор:")
        for exp_name, data in experiments_data.items():
            print(f"  🏷️  {exp_name}")
            print(f"     📅 {pd.Timestamp(data['timestamp'], unit='s')}")
            print(f"     🔖 Тег: {data['tag_used']}")
            print(f"     📝 {data['loss_function_info'][:100]}...")
            print()
            
    else:
        print("\n❌ Не найдено данных о функциях потерь")
        print("\n🔍 Попробуем извлечь все текстовые данные для отладки...")
        
        # Извлекаем все текстовые данные из первого эксперимента для отладки
        experiments = os.listdir("../runs/test/1/DeepSeek-V3.1")
        if experiments:
            first_exp = os.path.join("../runs/test/1/DeepSeek-V3.1", experiments[0])
            all_text_data = extract_all_text_data_from_experiment(first_exp)
            
            if all_text_data:
                print("📝 Все текстовые данные из первого эксперимента:")
                for tag, content in all_text_data.items():
                    print(f"\n🔖 Тег: {tag}")
                    print(f"📄 Содержимое: {content[:200]}...")
            else:
                print("❌ В первом эксперименте нет текстовых данных")

if __name__ == "__main__":
    main()

🔍 Поиск функций потерь в экспериментах PyTorch...

📋 Поиск всех текстовых тегов...
def loss_dinn(S_hat, S_pred, I_hat, I_pred, D_hat, D_pred, R_hat, R_pred, f1, f2, f3, f4, I_pred_last, train_size):
S_pred_slice = S_pred[:train_size]
I_pred_slice = I_pred[:train_size]
R_pred_slice = R_pred[:train_size]
D_pred_slice = D_pred[:train_size]
regul = 0.9
last_infected_penalty = 0.1
aggregation_func = torch.mean
norm_func = torch.square
term1 = aggregation_func(norm_func(S_hat - S_pred_slice))
term2 = aggregation_func(norm_func(I_hat - I_pred_slice))
term3 = aggregation_func(norm_func(D_hat - D_pred_slice))
term4 = aggregation_func(norm_func(R_hat - R_pred_slice))
term5 = aggregation_func(norm_func(f1))
term6 = aggregation_func(norm_func(f2))
term7 = aggregation_func(norm_func(f3))
term8 = aggregation_func(norm_func(f4))
loss = regul * (term1 + term2 + term3 + term4) + \
(1 - regul) * (term5 + term6 + term7 + term8) + \
last_infected_penalty * norm_func(I_pred_last-0)
return loss
def loss_din

In [1]:
import re

def extract_loss_functions_from_tensor(tensor_data):
    """
    Извлекает исходные функции потерь из сохраненного текста
    """
    # Декодируем если это tensor
    if hasattr(tensor_data, 'tensor_proto'):
        text_content = tensor_data.tensor_proto.string_val[0].decode('utf-8')
    else:
        text_content = tensor_data
    
    loss_func_before = None
    loss_func_after = None
    
    # Ищем исходную функцию потерь
    before_match = re.search(r'\*\*Исходная функция потерь:\*\*\s*```python\n(.*?)\n```', text_content, re.DOTALL)
    if before_match:
        loss_func_before = before_match.group(1).strip()
    
    # Ищем модифицированную функцию потерь
    after_match = re.search(r'\*\*Модифицированная функция потерь:\*\*\s*```python\n(.*?)\n```', text_content, re.DOTALL)
    if after_match:
        loss_func_after = after_match.group(1).strip()
    
    return remove_comments(loss_func_before), remove_comments(loss_func_after)


from codebleu import calc_codebleu

def remove_comments(code):
    """
    Удаляет комментарии из Python кода
    """
    # Удаляем блочные комментарии (тройные кавычки)
    code = re.sub(r'(\'\'\'[\s\S]*?\'\'\'|\"\"\"[\s\S]*?\"\"\")', '', code)
    
    # Удаляем однострочные комментарии
    code = re.sub(r'#.*$', '', code, flags=re.MULTILINE)
    
    # Удаляем пустые строки и лишние пробелы
    lines = code.split('\n')
    cleaned_lines = []
    for line in lines:
        stripped_line = line.strip()
        if stripped_line:  # Не добавляем пустые строки
            cleaned_lines.append(stripped_line)
    
    return '\n'.join(cleaned_lines)


def calculate_codebleu_with_library(loss_func_before, loss_func_after):
    """
    Calculate CodeBLEU using the codebleu package
    Requires: pip install codebleu
    """
    result = calc_codebleu(
            references=[[loss_func_before]],  # reference code
            hypothesis=[loss_func_after],     # generated code
            lang="python",
            weights=(0.25, 0.25, 0.25, 0.25)  # alpha, beta, gamma, theta
        )
        
    return {
            'codebleu': result['codebleu'],
            'ngram_match_score': result['ngram_match_score'],
            'weighted_ngram_match_score': result['weighted_ngram_match_score'],
            'syntax_match_score': result['syntax_match_score'],
            'dataflow_match_score': result['dataflow_match_score']
        }
        

ModuleNotFoundError: No module named 'codebleu'